In [ ]:
from pathlib import Path

RUN = "runs/detector-check-2026-11-11"

# Works when the kernel starts in the repository root or in notebooks/.
ROOT = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "stratscout").is_dir() and (p / "AGENTS.md").is_file()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from the CV-DL repository or its notebooks folder.")

run_dir = (ROOT / RUN).resolve()
if not (run_dir / "summary.json").is_file():
    raise FileNotFoundError(
        f"No finished detector check in: {run_dir}\n"
        "Change RUN to an existing check, or create one from the repository root with:\n"
        f'python -m stratscout detector-check --matches furia spirit --output "{RUN}"'
    )
print(f"Reading saved results from {run_dir}")

# Detector check results

1. Select the project's **.venv** Python kernel. If it is missing, install `ipykernel` in that environment.
2. Set `RUN` in the first cell to a finished `detector-check` folder, then run the cells from top to bottom.
3. Keep the figures on screen; this notebook does not write images or change results.

The check compares the colour detector's player positions with the true positions in the demo
files, on the **even rounds** of Furia and Spirit. Odd rounds were used only to line up demo
time and map coordinates. NaVi is never read. Demo positions are evaluation evidence only.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.switch_backend("module://matplotlib_inline.backend_inline")
plt.rcParams.update(
    {
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 11,
        "legend.fontsize": 9,
        "figure.dpi": 110,
    }
)

config = json.loads((run_dir / "config.json").read_text(encoding="utf-8"))
summary = json.loads((run_dir / "summary.json").read_text(encoding="utf-8"))
calibration = {
    c["match"]: c
    for c in (
        json.loads(p.read_text(encoding="utf-8")) for p in sorted(run_dir.glob("calibration-*.json"))
    )
}
players = pd.read_csv(run_dir / "players.csv")
detections = pd.read_csv(run_dir / "detections.csv")
mark = config["pass_mark"]
names = {m: m.split("-")[0].capitalize() + " vs Falcons" for m in calibration}
colours = {"T": "#d4a017", "CT": "#1f9bb4"}
print("Detector:", config["detector"])
print("Matches:", ", ".join(names.values()))
print(f"Verdict: {summary['verdict']}")
print("Calibration flags:", summary["calibration_flags"] or "none")

### Pass mark per team

Each row must pass all four checks. **Found** is the share of true players with a detection within
the match distance. **Extra** is the share of detections that match no true player.

In [ ]:
rows = []
for team in ("T", "CT"):
    s = summary["groups"]["pooled"][team]["all"]
    rows.append(
        {
            "Team": team,
            "Players": s["players"],
            "Found": f"{s['found_rate']:.1%} (need >= {mark['min_found']:.0%})",
            "Extra": f"{s['extra_rate']:.1%} (need <= {mark['max_extra']:.0%})",
            "Median px": f"{s['median_px']:.1f} (need <= {mark['median_px']:g})",
            "90th pct px": f"{s['p90_px']:.1f} (need <= {mark['p90_px']:g})",
            "Pass": "yes" if s["pass"] else "no",
        }
    )
pd.DataFrame(rows).set_index("Team")

### How far off are the detections?

Distance between each true player and its paired detection. Most of the mass should sit left of
the median line. The dashed lines are the pass mark's median and 90th-percentile limits.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
bins = np.arange(0, mark["error_gate_px"] + 1, 1)
for ax, team in zip(axes, ("T", "CT")):
    d = players.loc[players["team"] == team, "distance_px"].dropna()
    ax.hist(d, bins=bins, color=colours[team])
    for limit, style in ((mark["median_px"], "--"), (mark["p90_px"], ":")):
        ax.axvline(limit, color="black", linestyle=style, linewidth=1)
    ax.set_title(f"{team}: position error")
    ax.set_xlabel("Distance to true position (radar px)")
axes[0].set_ylabel("Player-frames")
plt.tight_layout()
plt.show()

### Where does the detector struggle?

Share of true players found in each kind of moment. Spawn stacks and overlapping icons are the
known weak spots; the bomb carrier is hidden under the C4 badge.

In [ ]:
moments = ["spawn stack", "overlapping", "isolated", "bomb carrier", "still", "moving"]
fig, ax = plt.subplots(figsize=(10, 3.8))
width = 0.38
for k, team in enumerate(("T", "CT")):
    groups = summary["groups"]["pooled"][team]
    shown = [m for m in moments if groups.get(m, {}).get("players")]
    x = np.array([moments.index(m) for m in shown]) + (k - 0.5) * width
    ax.bar(x, [groups[m]["found_rate"] for m in shown], width, label=team, color=colours[team])
    for xi, m in zip(x, shown):
        ax.text(xi, 0.02, str(groups[m]["players"]), ha="center", va="bottom", fontsize=8, rotation=90)
ax.axhline(mark["min_found"], color="black", linestyle="--", linewidth=1, label="pass mark")
ax.set_xticks(range(len(moments)), moments)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Players found")
ax.set_title("Found rate by moment (numbers = player-frames)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

### Does error grow with player speed?

If error rises sharply for fast players, part of it comes from timing (demo and video slightly out
of step) rather than from the detector itself.

In [ ]:
edges = [0, 20, 60, 120, 180, 240, 400]
fig, ax = plt.subplots(figsize=(7, 3.6))
for team in ("T", "CT"):
    sub = players[(players["team"] == team) & players["distance_px"].notna()]
    groups = sub.groupby(pd.cut(sub["speed"], edges), observed=True)["distance_px"].median()
    ax.plot(range(len(groups)), groups.values, marker="o", label=team, color=colours[team])
ax.set_xticks(range(len(edges) - 1), [f"{a}-{b}" for a, b in zip(edges, edges[1:])])
ax.set_xlabel("Player speed (game units per second)")
ax.set_ylabel("Median error (radar px)")
ax.set_title("Error by speed")
ax.legend()
plt.tight_layout()
plt.show()

### Calibration: lining up demo time with video time

Median paired distance for each tried time offset (odd rounds only). A single clear dip means the
demo and video were lined up confidently. The marked point is the offset used for scoring.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
for match, c in calibration.items():
    curve = np.array([(s, d if d is not None else np.nan) for s, d in c["offset_curve"]])
    ax.plot(curve[:, 0], curve[:, 1], label=names[match])
    ax.axvline(c["offset_s"], linestyle=":", color=ax.lines[-1].get_color())
ax.set_xlabel("Time offset tried (s)")
ax.set_ylabel("Median paired distance (px)")
ax.set_title("Time alignment search")
ax.legend()
plt.tight_layout()
plt.show()

pd.DataFrame(
    {
        names[m]: {
            "Round anchors, worst disagreement (s)": round(c["anchor_max_residual_s"], 2),
            "Offset used (s)": round(c["offset_s"], 2),
            "Demo/video speed ratio": round(c["time_scale"], 4),
            "Still, isolated players: median px": round(c["floor_median_px"], 2),
            "Still, isolated players: 90th pct px": round(c["floor_p90_px"], 2),
        }
        for m, c in calibration.items()
    }
)

### Where are players missed?

True positions of missed players (no detection within the match distance) on the radar crop.
Clusters show map areas where icons stack or overlap.

In [ ]:
missed = players[~players["found"]]
fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=True)
for ax, team in zip(axes, ("T", "CT")):
    found = players[(players["team"] == team) & players["found"]]
    lost = missed[missed["team"] == team]
    ax.scatter(found["x"], found["y"], s=1, color="lightgrey", label="found")
    ax.scatter(lost["x"], lost["y"], s=6, color=colours[team], label=f"missed ({len(lost)})")
    ax.set_xlim(0, 494)
    ax.set_ylim(500, 0)
    ax.set_aspect("equal")
    ax.set_title(f"{team}: missed players")
    ax.legend(loc="lower left", markerscale=3)
plt.tight_layout()
plt.show()